# Data Exploration Notebook for Approval Predict

## Objectives

Answer business requirement 1:
The client is interested in determining which applicant variables are most strongly correlated with the loan approval outcome. They want a ranked list of variables to be provided based on their relevance and impact.

## Inputs

* outputs/datasets/collection/loan_approval.csv

## Outputs

* generate code that answers business requirement 1 and can be used to build the Streamlit App.
* * Ranked list of influential variables for Business Requirement 1.
* Visualisations and metrics for the Streamlit app.

## Imports

In [ ]:
import os
from pathlib import Path
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
from feature_engine.discretisation import ArbitraryDiscretiser
from feature_engine.encoding import OneHotEncoder
warnings.filterwarnings("ignore")

## Change Working Directory

In [ ]:
current_dir = os.getcwd()
current_dir

os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

current_dir = os.getcwd()
current_dir

## Load Data

In this section, we load the loan approval dataset that was collected and saved in the Data Collection notebook. This dataset contains information about loan applicants including their financial details and the loan approval decision.

We will use this data to perform exploratory analysis and identify which variables are most strongly correlated with loan approval outcomes.

In [ ]:
root = current_dir
file_path = (
    Path(root)
    / "outputs"
    / "datasets"
    / "collection"
    / "loan_approval.csv"
)

if not file_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {file_path}"
    )

df = pd.read_csv(file_path).drop(['name'], axis=1)
df.head(3)


## Data Exploration

In this section, we use the ydata_profiling library to generate a comprehensive automated report of the dataset, providing insights into variable distributions, missing values, correlations, and data quality.

In [ ]:
from ydata_profiling import ProfileReport
pandas_report = ProfileReport(df=df, minimal=True)
pandas_report.to_notebook_iframe()

In [ ]:
approve_counts = df['loan_approved'].value_counts()
approve_counts

In [ ]:
percentage_approved = df['loan_approved'].value_counts(normalize=True) * 100
percentage_approved

### Initial Observations

**Target Variable Imbalance:**
- There is a moderate class imbalance in loan approvals, with rejections (False) having a higher proportion than approvals (True).
- This imbalance may cause some models to be biased towards predicting loan rejections.
- Class balancing techniques may need to be considered during model development.

**Variable Types:**
- The dataset contains 5 numerical variables, text variables, and one boolean target variable (`loan_approved`).

## Data Encoding

To enable correlation analysis, we need to convert categorical variables (`loan_approved` and `city`) to numeric values using One-Hot Encoding.

In [ ]:
encoder = OneHotEncoder(
    variables=df.columns[df.dtypes == "object"].to_list(),
    drop_last=False,
)
df_ohe = encoder.fit_transform(df)
print(df_ohe.shape)
df_ohe.head(3)

### Feature Engineering

Create a new feature `loan_to_income` to calculate the loan-to-income ratio, which represents the loan amount as a proportion of the applicant's income. This metric is commonly used in lending to assess affordability.

In [ ]:
df_ohe['loan_to_income'] = df_ohe['loan_amount'] / df_ohe['income']
df_ohe.head(10)

## Correlation Study

In this section, we calculate pairwise correlation coefficients between all numeric columns in the dataframe using two methods:
- **Spearman Correlation**: Measures monotonic relationships (variables that move in the same direction but not necessarily linearly)
- **Pearson Correlation**: Measures linear relationships (where 1 indicates perfect positive correlation and -1 indicates perfect negative correlation)

In [ ]:
df_ohe.dtypes

In [ ]:
corr_spearman = (
    df_ohe.corr(method="spearman")["loan_approved"]
    .sort_values(key=abs, ascending=False)[1:]
    .head(10)
)
corr_spearman

In [ ]:
corr_pearson = (
    df_ohe.corr(method="pearson")["loan_approved"]
    .sort_values(key=abs, ascending=False)[1:]
    .head(10)
)
corr_pearson

### Spearman Correlation Analysis

**Strong Positive Correlations:**
- `points` and `credit_score` have a strong positive correlation with `loan_approved`
- As points increase, loan approval likelihood increases significantly

**Weak Correlations:**
- `income` shows weak positive correlation
- `loan_amount` shows weak correlation
- `years_employed` has very weak correlation
- This suggests that higher income and lower loan amounts slightly increase approval chances, with years employed having minimal impact

**Loan-to-Income Ratio:**
- As the `loan_to_income` ratio increases, approval tends to decrease
- Higher ratios indicate less affordable loans, thus increasing rejection likelihood
- However, the correlation strength is relatively weak

**City Variable:**
- `city` has an extremely weak correlation with loan approval
- This variable has negligible effect on the approval decision

### Pearson Correlation Analysis

Pearson correlation shows a similar pattern to Spearman:
- `points` and `credit_score` demonstrate strong positive correlation with loan approval
- `income` shows very weak correlation
- This consistency across both correlation methods validates the importance of creditworthiness metrics (points and credit score) over income-based factors

### Feature Selection Decision

Based on the correlation analysis, we will drop the `city` feature from further analysis as it shows negligible correlation with loan approval and does not contribute meaningful predictive power to our model.

In [ ]:
if 'city' in df.columns:
    df = df.drop(['city'], axis=1)
df.head(4)

In [ ]:
vars_to_study = [
    "points",
    "credit_score",
    "income",
    "loan_amount",
    "years_employed",
    "loan_to_income",
]
vars_to_study

## Exploratory Data Analysis (EDA) on Selected Variables

Now that we've identified the most relevant features through correlation analysis, we'll perform detailed univariate and bivariate analysis on these selected variables to understand their distributions and relationships with loan approval.

In [ ]:
df_eda = df.filter(vars_to_study + ['loan_approved'])
df_eda.head(3)

## Variable Distributions by Loan Approval Status

The following visualisations show how each variable is distributed across approved and rejected loan applications using both histograms and boxplots.

In [ ]:
%matplotlib inline
sns.set_style('whitegrid')


def plot_numerical(df, vars_to_study, target_var):
    """
    Generate histogram and boxplot pairs for each numerical feature to
    compare distributions between target groups.
    """
    for col in vars_to_study:
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        sns.histplot(
            data=df, x=col, hue=target_var, kde=True, element="step",
            palette='Set1', ax=axes[0]
        )
        axes[0].set_title(f"{col} distribution by {target_var} (Histogram)")
        sns.boxplot(
            data=df, x=target_var, y=col, palette='Set2', width=0.4, ax=axes[1]
        )
        axes[1].set_title(f"{col} distribution by {target_var} (Boxplot)")
        plt.tight_layout()
        plt.show()


In [ ]:
target_var = 'loan_approved'
df['loan_to_income'] = df['loan_amount'] / df['income']

plot_numerical(df, vars_to_study, target_var)

### Points Distribution by Loan Approval Status

**Key Findings:**
- The boxplot shows applicants with approved loans have significantly higher points compared to those with rejected loans
- The median points for approved loans is around 75, while for rejected loans it's around 45
- Higher points are strongly associated with higher chances of loan approval

**Pattern:**
- Applicants with points below 60 are mostly rejected
- Applicants with points above ~70 are primarily approved
- This clear separation makes `points` a strong predictor of loan approval

### Credit Score Distribution by Loan Approval Status

**Key Findings:**
- The boxplot shows approved applicants have much higher credit scores than rejected applicants
- The median credit score for approved loans is around 700–750
- For rejected loans, the median is around 450–500
- Credit score appears to be a key predictor of loan approval

**Pattern:**
- Credit scores below 600 are more likely to be rejected
- Credit scores above 650 are mostly approved
- This represents one of the strongest predictive relationships in the dataset

### Income Distribution by Loan Approval Status

**Key Findings:**
- The boxplot shows approved applicants generally have higher incomes
- The median income for approved loans is around $100,000, compared to about $80,000 for rejected loans
- Income influences approval, but the difference is less pronounced than for credit score or points

**Pattern:**
- Applicants earning below ~$70k have more rejections
- Applicants earning above ~$90k tend to get approved
- Income plays a supporting role rather than being a primary decision factor

### Loan Amount Distribution by Loan Approval Status

**Key Findings:**
- The boxplot shows loan amounts are quite similar between approved and rejected groups
- The median loan amount for approved loans is slightly lower than for rejected loans
- Applicants requesting larger loans may be more likely to be rejected

**Pattern:**
- Applicants requesting larger loans are less likely to be approved
- Those asking for smaller to moderate loan amounts have a higher chance of approval
- The effect is relatively modest compared to creditworthiness factors

### Years Employed Distribution by Loan Approval Status

**Key Findings:**
- The boxplot shows years employed has a positive relationship with loan approval
- However, the median difference is small and there's considerable overlap between approved and rejected groups
- While employment stability helps, it's not a strong predictor

**Pattern:**
- Applicants with longer employment histories are slightly more likely to be approved
- However, rejections occur across all experience levels
- This variable has minimal discriminatory power for loan approval decisions

### Loan-to-Income Ratio Distribution by Loan Approval Status

**Key Findings:**
- The boxplot and histogram show that as loan amount becomes large relative to income, the likelihood of approval decreases
- Applicants with a lower loan-to-income ratio are more likely to have their loans approved
- Applicants requesting loans close to or exceeding their income level (outliers) are often rejected

**Pattern:**
- The presence of numerous outliers in the rejected group indicates riskier financial profiles
- Lenders are less likely to approve loans where the applicant's debt burden is too high relative to their income
- This metric serves as an important affordability and risk indicator

## Outlier Detection

Using the Interquartile Range (IQR) method to identify outliers in key variables. The IQR method defines outliers as values that fall below Q1 - 1.5×IQR or above Q3 + 1.5×IQR.

*Reference: [IQR Method Tutorial](https://www.youtube.com/shorts/SH7TPbT6zqE)*

In [ ]:
outliers_summary = {}

for col in ['credit_score', 'points', 'loan_to_income']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3-Q1

    upper_bound = Q3 + 1.5 * IQR
    lower_bound = Q1 - 1.5 * IQR

    mask = (df[col] < lower_bound) | (df[col] > upper_bound)
    outliers = df[mask]

    outliers_summary[col] = {
        'count': mask.sum(),
        'outliers': outliers.copy()
    }

    print(f"\n{col}:")
    print(f"  Number of outliers: {mask.sum()}")

### Outlier Analysis Results

The outlier detection identified that:
- `credit_score`: 0 outliers
- `points`: 0 outliers  
- `loan_to_income`: 115 outliers

The `loan_to_income` variable contains a significant number of outliers.

In [ ]:
percent = (mask.sum() / len(df) * 100).round(2)

In [ ]:
print(f"{col}: {mask.sum()} outliers ({percent}%)")

### Outlier Impact Assessment

The 115 outliers in `loan_to_income` represent **5.75% of the dataset**, which is a significant proportion. 

**Implications:**
- This justifies the use of robust scaling methods (like RobustScaler) instead of StandardScaler or MinMaxScaler when preparing data for modeling
- The presence of outliers will be considered when selecting appropriate preprocessing techniques

In [ ]:
sns.heatmap(df_eda.corr(numeric_only=True), annot=True, cmap='coolwarm')

### Correlation Heatmap Analysis

The correlation heatmap above provides a visual representation of the relationships between all numeric variables in our dataset. Key findings include:

**Strong Positive Correlations:**
- Points and Credit Score: These two variables are highly correlated with each other and with loan approval, making them the most important predictors.
- Both variables show strong positive correlation with `loan_approved`, confirming they are important factors in loan decisions.

**Weak to Moderate Correlations:**
- Income: Shows weak positive correlation with loan approval, suggesting it plays a secondary role.
- Years Employed: Very weak correlation with approval, indicating employment tenure has minimal impact.
- Loan Amount: Slightly negative correlation with approval, meaning larger loan requests are marginally riskier.


## Parallel Categories Plot

To better understand how multiple variables interact to influence loan approval, we'll discretize continuous variables into meaningful categories and visualize their relationships using a parallel categories plot.

**Discretization Bins:**
- `credit_score`: <580, 580-670, 670-740, 740-800
- `points`: <40, 40-60, 60-80, 80
- `loan_to_income`: <0.3, 0.3-0.5, 0.5-0.7, 0.7-1.0

In [ ]:
credit_map = [-np.inf, 580, 670, 740, 800, np.inf]
points_map = [-np.inf, 40, 60, 80, np.inf]
lti_map = [-np.inf, 0.3, 0.5, 0.7, 1.0, np.inf]

disc = ArbitraryDiscretiser(binning_dict={
    'credit_score': credit_map,
    'points': points_map,
    'loan_to_income': lti_map
})

df_eda['loan_to_income'] = df_eda['loan_amount'] / df_eda['income']
df_disc = disc.fit_transform(df_eda.copy())

### Label Mapping Function

The function below was created by CodeInstitute and adapted to perform label mapping for discretized numerical variables. 

In [ ]:
def make_label_map(binner_dict, variable):
    """
    Convert numeric bin indices from ArbitraryDiscretiser into readable
    category labels for visualisation.
    """
    bins = binner_dict[variable]
    df_disc["loan_approved"] = df_disc["loan_approved"].replace(
        {"No": "0", "Yes": "1"}
    )
    n_classes = len(bins) - 1
    classes_ranges = bins[1:-1]
    labels_map = {}
    for n in range(n_classes):
        if n == 0:
            labels_map[n] = f"<{classes_ranges[0]}"
        elif n == n_classes - 1:
            labels_map[n] = f"+{classes_ranges[-1]}"
        else:
            labels_map[n] = f"{classes_ranges[n-1]} to {classes_ranges[n]}"
    return labels_map


for var in ['credit_score', 'points', 'loan_to_income']:
    if var in disc.binner_dict_:
        df_disc[var] = df_disc[var].replace(
            make_label_map(disc.binner_dict_, var)
        )
    else:
        print(f"Skipping {var} - not found in discretiser")


In [ ]:
for col in ['credit_score', 'points', 'loan_to_income']:
    df_disc[col] = df_disc[col].astype(str)

df_disc['loan_approved'] = df_disc['loan_approved'].astype(int)

fig = px.parallel_categories(
    df_disc[['credit_score', 'points', 'loan_to_income', 'loan_approved']],
    color="loan_approved",
    color_continuous_scale=px.colors.sequential.Plasma
)

fig.show()

### Parallel Categories Plot Insights

The interactive visualisation reveals important multi-variable patterns:

**Creditworthiness Dominance:**
- `credit_score` and `points` are both strongly positively correlated with loan approval
- Applicants with low credit score AND low points have the highest rejection rates
- Applicants with high credit score AND high points almost always get approved

**Loan-to-Income Impact:**
- Applicants with lower loan-to-income ratios (specifically below 0.5) show a much higher proportion of approvals
- This indicates that lenders favor borrowers whose loan amount is small relative to their income
- As the loan-to-income ratio increases beyond 0.7, rejection rates increase significantly

**Combined Effect:**
- The best approval chances come from high creditworthiness (high points + high credit score) combined with reasonable affordability (low loan-to-income ratio)
- Poor creditworthiness alone is sufficient for rejection, regardless of affordability metrics

In [ ]:
ranking = pd.DataFrame({
    'Feature': corr_pearson.index,
    'Pearson_Corr': corr_pearson.values,
    'Spearman_Corr': corr_spearman.values
})
top_5 = ranking.head(5)[['Feature', 'Pearson_Corr']]
top_5

## Conclusions and Next Steps

* Applicants with high points are more likely to get approved.

* Applicants with high credit scores are more likely to get approved.

* Applicants with low points and low credit scores are more likely to be rejected.

* Income, loan amount, and years employed have minor effects on approval.

* City has almost no effect on approval.
  
* The parallel categories plot highlights a strong relationship between the loan-to-income ratio and loan approval outcomes.
  
* The best features to analyse include points, credit score and loan to income.

Next steps:

* Use points, credit score and loan to income as primary features in predictive models.
* Consider using a RobustScaler when preparing for linear regression rather than StandardScaler or MinMax Scale due to significant number of outliers in the loan_to_amount variable.